# Stage 5: Model selection

This notebook reads the saved cross-validation results and chooses a model using PR-AUC first. It does not refit models or change the results file.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_PATH = PROJECT_ROOT / 'reports' / 'model_comparison.csv'
results = pd.read_csv(RESULTS_PATH)

print(f'Loaded {len(results)} model results from {RESULTS_PATH}')
print('The CSV is only read; this notebook does not write it.')

## Models sorted by cross-validation PR-AUC

In [ ]:
# The primary metric is PR-AUC because the subscribed class is the minority class.
summary = results.assign(
    pr_auc_gap=results['train_pr_auc_mean'] - results['pr_auc_mean']
).sort_values('pr_auc_mean', ascending=False)

display_columns = [
    'name',
    'pr_auc_mean', 'pr_auc_std',
    'roc_auc_mean', 'roc_auc_std',
    'f1_mean', 'f1_std',
    'precision_mean', 'precision_std',
    'recall_mean', 'recall_std',
    'pr_auc_gap', 'fit_time_mean',
]
display(summary[display_columns].round(4))

## Cross-validation PR-AUC with uncertainty

In [ ]:
# Error bars show the standard deviation across the five validation folds.
plot_data = summary.sort_values('pr_auc_mean')
plt.figure(figsize=(10, 6))
plt.barh(
    plot_data['name'],
    plot_data['pr_auc_mean'],
    xerr=plot_data['pr_auc_std'],
    color='#2f6690',
    alpha=0.9,
    capsize=4,
)
plt.xlabel('Mean CV PR-AUC')
plt.ylabel('')
plt.title('Model comparison by cross-validation PR-AUC')
plt.xlim(left=0)
plt.tight_layout()
plt.show()

## Ranking and close results

Rank models by mean CV PR-AUC, but do not overstate small differences. A pair is flagged below when the absolute difference between their mean PR-AUC values is smaller than the larger of their fold standard deviations. Such a pair is close enough that the simpler, faster, or easier-to-explain model may be preferable.

In [ ]:
# Print the ranking and identify pairs whose difference is smaller than one standard deviation.
for rank, (_, row) in enumerate(summary.iterrows(), start=1):
    print(f'{rank}. {row["name"]}: PR-AUC {row["pr_auc_mean"]:.4f} +/- {row["pr_auc_std"]:.4f}')

close_pairs = []
for left_index in range(len(summary)):
    for right_index in range(left_index + 1, len(summary)):
        left = summary.iloc[left_index]
        right = summary.iloc[right_index]
        difference = abs(left['pr_auc_mean'] - right['pr_auc_mean'])
        one_std = max(left['pr_auc_std'], right['pr_auc_std'])
        if difference < one_std:
            close_pairs.append({
                'model_a': left['name'],
                'model_b': right['name'],
                'mean_difference': difference,
                'larger_std': one_std,
            })

close_pairs = pd.DataFrame(close_pairs).sort_values('mean_difference')
if close_pairs.empty:
    print('No pair has a mean PR-AUC difference smaller than one standard deviation.')
else:
    print('Pairs whose difference is smaller than one standard deviation:')
    display(close_pairs.head(10).round(4))

print('Selection rule: start with the highest PR-AUC, then consider uncertainty, the PR-AUC gap, fit time, and explainability before finalising the model.')

## Decision

The first-ranked model by mean CV PR-AUC is the leading candidate. The standard-deviation comparison prevents treating a tiny numerical difference as a meaningful improvement. A large train-versus-CV PR-AUC gap is a warning sign for overfitting, while fit time and explainability matter for a model that must be demonstrated and served.